In [ ]:
import numpy as np
from src.environments.simple_trading_env import SimpleTradingEnv
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import PPO
from src.utils.indicator_utils import add_indicators
import pandas as pd
import torch
import matplotlib.pyplot as plt

symbol = 'BTCUSDT'
timeframe = '5m'
data_path = f'data/binance-{symbol}-{timeframe}.pkl'
df = pd.read_pickle(data_path)

print(f'Loaded {len(df)} rows for {symbol} {timeframe}')


# Test on unseen data after training set
total_timesteps = 2000
test_data = df.iloc[10000:10000 + total_timesteps]

# Test on a single environment
test_env = SimpleTradingEnv(test_data, device="cuda")
test_env = Monitor(test_env)
test_env = DummyVecEnv([lambda: test_env])
#test_env = VecNormalize.load("vecnormalize_normalized.pkl", test_env)
#test_env.training = False
#test_env.norm_reward = False

# Load your trained model
model = PPO.load("trading_bot", env=test_env, device="cuda")

# === FULLY DYNAMIC FEATURE EXTRACTION ===
# Auto-discover ALL feature modules from the extractor
extractor = model.policy.features_extractor

# Pattern matching for known feature types (encoders, outputs, CNNs, transformers)
HOOKABLE_PATTERNS = [
    '_cnn',       # CNNs (spatial, temporal, divergence)
    '_output',    # Output layers
    '_encoder',   # Encoders (momentum, session, account, position)
    '_transformer',  # Transformers (if you want to hook them)
]

# Auto-discover all hookable modules
available_features = {}
print("Scanning extractor for hookable modules...")
print(f"Extractor type: {type(extractor).__name__}\n")

for attr_name in dir(extractor):
    # Skip private/magic methods and non-module attributes
    if attr_name.startswith('_'):
        continue
    
    # Check if it matches our patterns
    is_hookable = any(pattern in attr_name for pattern in HOOKABLE_PATTERNS)
    
    if is_hookable:
        attr = getattr(extractor, attr_name)
        # Verify it's actually a module (not a method or property)
        if isinstance(attr, torch.nn.Module):
            # Create a clean display name
            display_name = attr_name.replace('_', ' ').title().replace(' ', '_')
            available_features[attr_name] = display_name
            print(f"  ✓ Found: {attr_name:30} -> {display_name}")

print(f"\nDiscovered {len(available_features)} feature groups to track")

# Initialize tracking dictionary dynamically
feature_activations = {display_name: [] for display_name in available_features.values()}

obs = test_env.reset()
total_reward = 0
steps = 0
last_env = None

# Register hooks dynamically
activations_storage = []

def make_hook(name):
    def hook(module, input, output):
        # Handle different output types (tuple, tensor, etc.)
        if isinstance(output, tuple):
            output = output[0]
        if isinstance(output, torch.Tensor):
            activations_storage.append({
                'name': name,
                'magnitude': output.abs().mean().item()
            })
    return hook

# Register hooks for all discovered features
hooks = []
for module_name, display_name in available_features.items():
    module = getattr(extractor, module_name)
    hook = module.register_forward_hook(make_hook(display_name))
    hooks.append(hook)

print(f"\nRegistered {len(hooks)} hooks for feature tracking")
print(f"\nRunning evaluation with feature tracking...\n")

while True:
    activations_storage.clear()
    action, _states = model.predict(obs, deterministic=True)
    
    # Store activations from this step
    for act in activations_storage:
        feature_activations[act['name']].append(act['magnitude'])
    
    obs, reward, done, info = test_env.step(action)
    total_reward += reward[0]
    steps += 1
    if done[0]:
        break
    last_env = test_env.envs[0].env.history[-1]

# Clean up hooks
for hook in hooks:
    hook.remove()

print(f"\n{'='*60}")
print(f"Total reward: {total_reward:.4f}")
print(f"Steps: {steps}")

# Calculate average activation per feature group
print(f"\n{'='*60}")
print("Feature Group Importance (Average Activation Magnitude):")
print(f"{'='*60}")

avg_activations = {}
for name, values in feature_activations.items():
    if len(values) > 0:
        avg = np.mean(values)
        avg_activations[name] = avg
        print(f"{name:35}: {avg:.4f} ({len(values)} activations)")
    else:
        print(f"{name:35}: NO ACTIVATIONS (hook not triggered)")

# Visualize feature importance
if avg_activations:
    plt.figure(figsize=(16, 8))
    names = list(avg_activations.keys())
    values = list(avg_activations.values())
    
    colors = ['#FF6B6B', '#4ECDC4', '#FF9F43', '#FFA502', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F', '#BB8FCE', '#95E1D3', '#F38181', '#85C1E2', '#C7CEEA']
    
    plt.bar(names, values, color=colors[:len(names)])
    plt.xlabel('Feature Group', fontsize=12)
    plt.ylabel('Average Activation Magnitude', fontsize=12)
    plt.title(f'Feature Group Importance - Which Features Drive Trading Decisions? ({len(names)} Groups)', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

if last_env:
    perf = last_env.get('performance', {})
    print(f"\n{'='*60}")
    print("Performance Metrics:")
    print(f"{'='*60}")
    for k, v in perf.items():
        print(f"{k:20}: {v}")

    trades = last_env.get('trades', [])
    total_trades = len(trades)
    
    print(f"\n{'='*60}")
    print(f"Trades Summary ({total_trades} total):")
    print(f"{'='*60}")
    
    if total_trades > 0:
        # Count trade outcomes
        tp_count = sum(1 for t in trades if 'TP' in t.get('reason', ''))
        sl_count = sum(1 for t in trades if 'SL' in t.get('reason', ''))
        dir_change = total_trades - tp_count - sl_count
        
        print(f"TP Exits: {tp_count} ({tp_count/total_trades*100:.1f}%)")
        print(f"SL Exits: {sl_count} ({sl_count/total_trades*100:.1f}%)")
        print(f"Direction Change: {dir_change} ({dir_change/total_trades*100:.1f}%)")
        print(f"Average Trade Duration: {np.mean([t.get('duration', 0) for t in trades]):.1f} steps")
        print(f"Trading Frequency: 1 trade every {steps/total_trades:.0f} steps")
        print(f"{'='*60}\n")
    
        # Show first 20 trades only for quick review
        for i, trade in enumerate(trades[:20], 1):
            direction_str = "LONG" if trade['direction'] == 1 else "SHORT"
            pnl = trade.get('pnl', 0)
            pnl_pct = trade.get('pnl_percent', 0) * 100
            reason = trade.get('reason', 'N/A')
            duration = trade.get('duration', 0)
            print(f"Trade {i:2}: {direction_str:5} | PnL: {pnl:8.2f} ({pnl_pct:+6.2f}%) | Duration: {duration:3} steps | Exit: {reason}")
        
        if total_trades > 20:
            print(f"... and {total_trades - 20} more trades")
    else:
        print("No trades executed!")
else:
    print("No environment info available.")

Loaded 264323 rows for BTCUSDT 5m


Loaded 264323 rows for BTCUSDT 5m


ValueError: Observation spaces do not match: Dict('account_state': Box(-inf, inf, (288, 4), float32), 'macd_divergence': Box(-inf, inf, (288, 3), float32), 'momentum_oscillators': Box(-inf, inf, (288, 2), float32), 'performance_metrics': Box(-inf, inf, (288, 7), float32), 'position_info': Box(-inf, inf, (288, 7), float32), 'price_context': Box(-inf, inf, (288, 12), float32), 'price_ohlc_spatial': Box(-inf, inf, (288, 4), float32), 'price_ohlc_temporal': Box(-inf, inf, (288, 4), float32), 'rsi_divergence': Box(-inf, inf, (288, 2), float32), 'trading_sessions': Box(0.0, 1.0, (288, 3), float32), 'trend_indicators': Box(-inf, inf, (288, 10), float32), 'volume_profile': Box(-inf, inf, (288, 26), float32), 'vp_distribution': Box(0.0, 1.0, (288, 54), float32)) != Dict('account_state': Box(-inf, inf, (288, 1), float32), 'macd_divergence': Box(-inf, inf, (288, 3), float32), 'momentum_oscillators': Box(-inf, inf, (288, 2), float32), 'position_info': Box(-inf, inf, (288, 2), float32), 'price_cont

In [ ]:
import numpy as np
from src.environments.simple_trading_env import SimpleTradingEnv
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from matplotlib import widgets
from stable_baselines3 import PPO
import pandas as pd
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from tqdm import tqdm
from src.utils.indicator_utils import add_indicators
from IPython.display import display, clear_output

symbol = 'BTCUSDT'
timeframe = '5m'
data_path = f'data/binance-{symbol}-{timeframe}.pkl'
df = pd.read_pickle(data_path)

print(f'Loaded {len(df)} rows for {symbol} {timeframe}')

""" total_timesteps = 500
i = np.random.randint(0, len(df) - total_timesteps)
test_data = df.iloc[i:i + total_timesteps] """

total_timesteps = 3000
test_data = df.iloc[220000:220000+total_timesteps]

# Test on a single environment
eval_env = SimpleTradingEnv(test_data, device="cuda", render_mode='plot')
eval_env = DummyVecEnv([lambda: eval_env])
#eval_env = VecNormalize.load("vecnormalize_normalized.pkl", eval_env)
#eval_env.training = False
#eval_env.norm_reward = False  # if you want raw rewards during evaluation

# Load your trained model
model = PPO.load("trading_bot", env=eval_env)
#model = PPO.load("models/best_model/best_model.zip", env=eval_env)


obs = eval_env.reset()
done = False
total_reward = 0.0
frames = []
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = eval_env.step(action)
    total_reward += reward[0]
    frame = eval_env.envs[0].render()  # Use your env's render

    if not done and frame is not None:
        frames.append(frame)


print(f"Evaluation finished. Total reward: {total_reward}")
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

# Display a slider to view frames (images) generated by render()
# Display a slider to view frames (images) generated by render()
if 'frames' in locals() and len(frames) > 0:
    # Create output widget to control display
    output = widgets.Output()

    # Current index tracker
    current_idx = [0]  # Use list to allow modification in nested function

    def show_frame(idx):
        with output:
            clear_output(wait=True)
            plt.figure(figsize=(15,12))
            plt.imshow(frames[idx])
            plt.axis('off')
            plt.title(f'Frame {idx+1} / {len(frames)}')
            plt.show()

    def on_previous(b):
        if current_idx[0] > 0:
            current_idx[0] -= 1
            slider.value = current_idx[0]
            show_frame(current_idx[0])

    def on_next(b):
        if current_idx[0] < len(frames) - 1:
            current_idx[0] += 1
            slider.value = current_idx[0]
            show_frame(current_idx[0])

    def on_slider_change(change):
        current_idx[0] = change['new']
        show_frame(current_idx[0])

    # Create widgets
    prev_button = widgets.Button(description='Previous', button_style='info', icon='arrow-left')
    next_button = widgets.Button(description='Next', button_style='info', icon='arrow-right')
    slider = widgets.IntSlider(value=0, min=0, max=len(frames)-1, step=1, description='Frame:')

    # Attach event handlers
    prev_button.on_click(on_previous)
    next_button.on_click(on_next)
    slider.observe(on_slider_change, names='value')

    # Layout
    button_box = widgets.HBox([prev_button, next_button])
    controls = widgets.VBox([slider, button_box])

    # Display
    display(controls)
    display(output)

    # Show initial frame
    show_frame(0)
else:
    print('No frames to display. Run the evaluation cell to generate frames.')

In [ ]:
# Deep feature analysis
print("\n" + "="*60)
print("DEEP FEATURE ANALYSIS")
print("="*60)

extractor = model.policy.features_extractor

# 1. Check parameter statistics
print("\nParameter Statistics (mean absolute value):")
for name, module in extractor.named_modules():
    if hasattr(module, 'weight') and module.weight is not None:
        weight_mean = module.weight.abs().mean().item()
        weight_std = module.weight.std().item()
        print(f"{name:30} | mean: {weight_mean:.6f} | std: {weight_std:.6f}")

# 2. Check if parameters are frozen
print("\nParameter Requires Grad:")
for name, param in extractor.named_parameters():
    if 'temporal' in name or 'spatial' in name or 'vp_bins' in name:
        print(f"{name:50} | requires_grad: {param.requires_grad} | shape: {list(param.shape)}")

# 3. Check feature variance (diversity)
print("\nFeature Output Variance (over 100 steps):")
test_env.reset()
feature_outputs = {k: [] for k in feature_activations.keys()}

for _ in range(100):
    obs = test_env.reset()
    with torch.no_grad():
        features = extractor(obs)
    
    # This won't work directly - need to modify hooks
    # But shows what we need to track